In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import FashionMNIST
from torchvision.transforms import ToTensor
import torchvision.utils as vutils
import matplotlib.pyplot as plt

# Define the modified VanillaAutoencoder
class VanillaAutoencoder(nn.Module):
    def __init__(self, input_size, latent_size, aux_input_size):
        super(VanillaAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_size + aux_input_size, 128),
            nn.ReLU(),
            nn.Linear(128, latent_size)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_size, 128),
            nn.ReLU(),
            nn.Linear(128, input_size)
        )

    def forward(self, input_data, aux_input):
        concatenated_input = torch.cat((input_data, aux_input), dim=1)
        latent_representation = self.encoder(concatenated_input)
        reconstructed_data = self.decoder(latent_representation)
        return reconstructed_data

def reconstruct_sample(autoencoder, sample_data, aux_input):
    reconstructed_sample = autoencoder(sample_data, aux_input)
    return reconstructed_sample

# Set random seed for reproducibility
torch.manual_seed(42)

# Hyperparameters
input_size = 784
latent_size = 32
aux_input_size = 1
batch_size = 64
num_epochs = 20
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load FashionMNIST dataset
dataset = FashionMNIST(root="./data", train=True, transform=ToTensor(), download=True)
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Create an instance of the modified VanillaAutoencoder
autoencoder = VanillaAutoencoder(input_size, latent_size, aux_input_size).to(device)

# Define the loss function
loss_function = nn.MSELoss()

# Define the optimizer
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=0.001)

# Training loop
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(data_loader):
        images = images.view(-1, input_size).to(device)
        labels = labels.float().view(-1, aux_input_size).to(device)

        # Forward pass
        reconstructed_images = autoencoder(images, labels)

        # Compute the reconstruction loss
        loss = loss_function(reconstructed_images, images)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i+1) % 200 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(data_loader)}], Loss: {loss.item():.4f}")

    # Save a generated sample for visualization
    with torch.no_grad():
        sample_labels = torch.tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]).float().view(-1, aux_input_size).to(device)
        generated_samples = autoencoder.decoder(torch.randn(10, latent_size).to(device)).cpu().view(-1, 1, 28, 28)
        vutils.save_image(generated_samples, f"generated_samples_epoch_{epoch+1}.png", normalize=True, nrow=10)

print("Training completed!")

# Choose a random sample for reconstruction and visualization
sample_index = 0
sample_data, sample_label = dataset[sample_index]
sample_data = sample_data.view(1, input_size).to(device)
sample_label = sample_label.float().view(1, aux_input_size).to(device)

# Reconstruct the sample
reconstructed_sample = reconstruct_sample(autoencoder, sample_data, sample_label)

# Visualize the original and reconstructed samples
sample_data = sample_data.cpu().view(1, 28, 28)
reconstructed_sample = reconstructed_sample.cpu().view(1, 28, 28)

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.title("Original Sample")
plt.imshow(sample_data[0], cmap="gray")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.title("Reconstructed Sample")
plt.imshow(reconstructed_sample[0], cmap="gray")
plt.axis("off")

plt.tight_layout()
plt.show()
